# FermiNet-inspired representation design -- MultipoleNet

In [1]:
import copy
from typing import Dict, List, Optional, Union

import torch
import torch.nn as nn
from loguru import logger as log

from modelforge.potential.aimnet2 import AimNet2Core, AIMNet2Representation, AIMNet2InteractionModule, mlp_init
from modelforge.potential.parameters import ActivationFunctionConfig
from modelforge.potential.neighbors import NeighborListForTraining, PairlistData
from modelforge.utils.prop import NNPInput

from e3nn import o3
from e3nn.nn import FullyConnectedNet, Gate
from e3nn.math import soft_one_hot_linspace

## NNP Modules

In [2]:
from modelforge.potential.multipolenet import (
    MultipoleNetCore,
    MultipoleInteractionModule,
    MultipoleRepresentation,
    MultipoleReadout,
)

## Step-by-step overview

In [3]:
Z = torch.tensor([26, 8, 8])                                    # Fe, O, O
pos = torch.tensor(
    [
        [0., 0., 0.],
        [1.8, 0., 0.],
        [-1.8, 0., 0.],
    ], 
    requires_grad=True,
)
spin_multiplicity, q = 5, +1

In [4]:
data = NNPInput(
    atomic_numbers=Z,
    positions=pos,
    atomic_subsystem_indices=torch.tensor([0, 0, 0]),
    per_system_total_charge=torch.tensor([q]).unsqueeze(1),
    per_system_spin_multiplicity=torch.tensor([spin_multiplicity]),
)

In [5]:
NNPInput

modelforge.utils.prop.NNPInput

In [6]:
neighborlist_module = NeighborListForTraining(
    local_cutoff=5,
)
neighborlist = neighborlist_module(
    data,
)

In [7]:
neighborlist

PairlistOutputs(local_cutoff=PairlistData(pair_indices=tensor([[0, 0, 1, 1, 2, 2],
        [1, 2, 0, 2, 0, 1]]), d_ij=tensor([[1.8000],
        [1.8000],
        [1.8000],
        [3.6000],
        [1.8000],
        [3.6000]], grad_fn=<IndexBackward0>), r_ij=tensor([[ 1.8000,  0.0000,  0.0000],
        [-1.8000,  0.0000,  0.0000],
        [-1.8000,  0.0000,  0.0000],
        [-3.6000,  0.0000,  0.0000],
        [ 1.8000,  0.0000,  0.0000],
        [ 3.6000,  0.0000,  0.0000]], grad_fn=<IndexBackward0>), only_unique_pairs=tensor(False)), vdw_cutoff=PairlistData(pair_indices=tensor([], size=(2, 0), dtype=torch.int64), d_ij=tensor([], size=(0, 1)), r_ij=tensor([], size=(0, 3)), only_unique_pairs=tensor(False)), electrostatic_cutoff=PairlistData(pair_indices=tensor([], size=(2, 0), dtype=torch.int64), d_ij=tensor([], size=(0, 1)), r_ij=tensor([], size=(0, 3)), only_unique_pairs=tensor(True)))

In [8]:
core = MultipoleNetCore(
    number_of_radial_basis_functions=16,
    number_of_interaction_modules=3,
    number_of_monopole_dimensions=64,
    number_of_dipole_dimensions=16,
    number_of_quadrupole_dimensions=8,
    maximum_interaction_radius=5,
    number_of_radial_basis_module_dimensions=8,
    activation_function_parameter={
        "activation_function_name": "SiLU",
        "activation_function": nn.SiLU(),
    },
    readout_hidden_features=64,
)

2026-10-25 16:22:24.844 | DEBUG    | modelforge.potential.multipolenet:__init__:46 - Initializing the MultipoleNet architecture.


In [9]:
core

MultipoleNetCore(
  (activation_function): SiLU()
  (representation_module): MultipoleRepresentation(
    (activation_function): SiLU()
    (conditioning): Linear(in_features=2, out_features=64, bias=True)
    (interaction_layers): ModuleList(
      (0): MultipoleInteractionModule(
        (activation_function): SiLU()
        (gate): Gate (88x0e+16x1o+8x2e -> 64x0e+16x1o+8x2e)
        (tensor_product): FullyConnectedTensorProduct(64x0e x 1x0e+1x1o+1x2e -> 88x0e+16x1o+8x2e | 7168 paths | 7168 weights)
        (radial_basis_module): FullyConnectedNet[16, 8, 7168]
        (self_interaction): Linear(64x0e -> 88x0e+16x1o+8x2e | 5632 weights)
      )
      (1-2): 2 x MultipoleInteractionModule(
        (activation_function): SiLU()
        (gate): Gate (88x0e+16x1o+8x2e -> 64x0e+16x1o+8x2e)
        (tensor_product): FullyConnectedTensorProduct(64x0e+16x1o+8x2e x 1x0e+1x1o+1x2e -> 88x0e+16x1o+8x2e | 10176 paths | 10176 weights)
        (radial_basis_module): FullyConnectedNet[16, 8, 10176]
 

In [10]:
results = core(data, neighborlist.local_cutoff)

## Use MultipoleNet Representation

In [11]:
results

{'per_atom_charge_multipole': tensor([[ 0.2096,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000, -2.4757,  0.0000,
          -4.2881],
         [ 0.3952, -1.4320,  0.0000,  0.0000,  0.0000,  0.0000, -2.1734,  0.0000,
          -3.7645],
         [ 0.3952,  1.4320,  0.0000,  0.0000,  0.0000,  0.0000, -2.1734,  0.0000,
          -3.7645]], grad_fn=<CopySlices>),
 'per_atom_spin_multipole': tensor([[ 1.5605,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0125,  0.0000,
           0.0216],
         [ 1.2198,  0.0597,  0.0000,  0.0000,  0.0000,  0.0000,  0.4251,  0.0000,
           0.7362],
         [ 1.2198, -0.0597,  0.0000,  0.0000,  0.0000,  0.0000,  0.4251,  0.0000,
           0.7362]], grad_fn=<CopySlices>),
 'per_atom_scalar_representation': tensor([[0.2096, 0.0000, 4.9515, 1.5605, 0.0000, 0.0249],
         [0.3952, 1.4320, 4.3469, 1.2198, 0.0597, 0.8501],
         [0.3952, 1.4320, 4.3469, 1.2198, 0.0597, 0.8501]],
        grad_fn=<CatBackward0>),
 'atomic_subsystem_indices': tensor([0,

We can use the per_atom_scalar_representation as the representation.

In [12]:
results["per_atom_scalar_representation"]

tensor([[0.2096, 0.0000, 4.9515, 1.5605, 0.0000, 0.0249],
        [0.3952, 1.4320, 4.3469, 1.2198, 0.0597, 0.8501],
        [0.3952, 1.4320, 4.3469, 1.2198, 0.0597, 0.8501]],
       grad_fn=<CatBackward0>)

In [13]:
class MultipoleRep_AimNet2Core(torch.nn.Module):
    def __init__(
        self,
        featurization: Dict[str, Dict[str, int]],
        number_of_radial_basis_functions: int,
        number_of_vector_features: int,
        number_of_interaction_modules: int,
        interaction_module_hidden_layers: List[List[int]],
        output_module_hidden_layers: List[int],
        activation_function_parameter: Dict[str, str],
        predicted_properties: List[str],
        predicted_dim: List[int],
        maximum_interaction_radius: float,
    ) -> None:
        """
        Core architecture of the AimNet2 model for molecular property
        prediction.

        Parameters
        ----------
        featurization : Dict[str, Dict[str, int]]
            Configuration dictionary specifying feature details for atomic
            embeddings.
        number_of_radial_basis_functions : int
            Number of radial basis functions used in the radial symmetry
            function.
        number_of_vector_features : int
            Number of vector features used in the model, which determines the
            dimensionality of the angular symmetry functions.
        number_of_interaction_modules : int
            Number of interaction modules in the model, determining the depth of
            message passing.
        interaction_module_hidden_layers : List[List[int]]
            List of hidden layer sizes for each interaction module. Note, this requires at least one value to be defined
            as this specifies the output size of the first layer in the interaction module and input size of the final layer
        output_module_hidden_layers : List[int]
            List of hidden layer sizes for the output modules, which are used to
            compute the final predictions from the atomic embeddings.
        activation_function_parameter : Dict[str, str]
            Configuration of activation functions used across the model.
        predicted_properties : List[str]
            List of properties that the model is predicting (e.g., energy,
            forces).
        predicted_dim : List[int]
            The dimensionality of each predicted property.
        maximum_interaction_radius : float
            The cutoff radius for atomic interactions in the model.
        """

        super().__init__()

        log.debug("Initializing the AimNet2 architecture.")

        self.activation_function = activation_function_parameter["activation_function"]
        # self.activation_function = activation_function_parameter.activation_function


        # Initialize representation block
        self.representation_module = core
        self.aimnet2_representation_module =  AIMNet2Representation(
            maximum_interaction_radius,
            number_of_radial_basis_functions,
            featurization_config=featurization,
        )
        number_of_per_atom_features = int(
            featurization["atomic_number"]["number_of_per_atom_features"]
        )

        self.agh = nn.Parameter(
            torch.randn(
                number_of_per_atom_features,  # F_atom
                number_of_radial_basis_functions,  # G
                number_of_vector_features,  # H
            )
        )
        # shape(nr_of_angular_symmetry_functions,nr_of_radial_symmetry_functions,nr_of_vector_features)

        # Define interaction modules for message passing
        self.interaction_modules = torch.nn.ModuleList(
            [
                AIMNet2InteractionModule(
                    number_of_per_atom_features=number_of_per_atom_features,
                    number_of_radial_basis_functions=number_of_radial_basis_functions,
                    number_of_vector_features=number_of_vector_features,
                    number_of_charge_channels=1,
                    hidden_layers=interaction_module_hidden_layers[i],
                    activation_function=self.activation_function,
                    is_first_module=(i == 0),
                )
                for i in range(number_of_interaction_modules)
            ]
        )
        # Define output layers to calculate per-atom predictions
        self.output_layers = nn.ModuleDict()
        for property, dim in zip(predicted_properties, predicted_dim):
            self.output_layers[property] = mlp_init(
                n_in_features=number_of_per_atom_features,
                n_out_features=int(dim),
                activation_function=self.activation_function,
                hidden_layers=output_module_hidden_layers,
            )

        from modelforge.potential.processing import ChargeConservation

        self.charge_conservation = ChargeConservation()

        from modelforge.potential.processing import CoulombPotential

        # note this will need to change when we switch to supporting multiple cutoffs
        self.coulomb_potential = CoulombPotential(cutoff=maximum_interaction_radius)

    def compute_properties(
        self,
        data: NNPInput,
        pairlist: PairlistData,
    ) -> Dict[str, torch.Tensor]:
        """
        Calculate the requested properties for a given input batch.

        Parameters
        ----------
        data : NNPInput
            The input data for the model.
        pairlist: PairlistData
            The output from the pairlist module.
        Returns
        -------
        Dict[str, torch.Tensor]
            The calculated per-atom scalar representations and atomic subsystem
            indices.
        """

        rep = self.representation_module(data, pairlist)

        aimnet2_rep = self.aimnet2_representation_module(data, pairlist)
        rep["f_ij"] = aimnet2_rep["f_ij"]
        rep["f_cutoff"] = aimnet2_rep["f_cutoff"]
        
        atomic_embedding = rep["per_atom_scalar_representation"]
        r_ij, d_ij, f_ij, f_cutoff = (
            pairlist.r_ij,
            pairlist.d_ij,
            rep["f_ij"],
            rep["f_cutoff"],
        )
        # Scalar Gaussian expansion for radial terms
        gs = f_ij * f_cutoff  # Shape: (number_of_pairs, G)
        # Unit direction vectors
        u_ij = r_ij / d_ij
        # Compute gv with shape (number_of_pairs, 3, G)
        gv = u_ij.unsqueeze(-1) * gs.unsqueeze(1)  # Broadcasting over G

        # Atomic embedding "a" Eqn. (3)
        partial_charges = torch.zeros(
            (atomic_embedding.shape[0], 1), device=atomic_embedding.device
        )

        # Perform message passing using interaction modules
        for i, interaction in enumerate(self.interaction_modules):

            delta_a, delta_q, f = interaction(
                atomic_embedding,
                partial_charges,
                pairlist.pair_indices,
                gs,
                gv,
                self.agh,
            )

            # Update atomic embeddings
            atomic_embedding = atomic_embedding + delta_a

            # Apply scaling factor `f` to `delta_q`
            scaled_delta_q = f * delta_q

            # Update partial charges
            if i == 0:
                partial_charges = scaled_delta_q  # Initialize charges
            else:
                partial_charges = partial_charges + scaled_delta_q  # Incremental update

            partial_charges = self.charge_conservation(
                {
                    "per_atom_charge": partial_charges,
                    "per_system_total_charge": data.per_system_total_charge.to(
                        dtype=torch.float32
                    ),
                    "atomic_subsystem_indices": data.atomic_subsystem_indices.to(
                        dtype=torch.int64
                    ),
                }
            )["per_atom_charge"]

        # check that none of the tensors are NaN
        if torch.isnan(atomic_embedding).any():
            raise ValueError("NaN values detected in atomic embeddings.")
        if torch.isnan(partial_charges).any():
            raise ValueError("NaN values detected in partial charges.")

        return {
            "per_atom_scalar_representation": atomic_embedding,
            "atomic_subsystem_indices": data.atomic_subsystem_indices,
            "atomic_numbers": data.atomic_numbers,
            "per_atom_charge": partial_charges,
        }

    def forward(
        self,
        data: NNPInput,
        pairlist_output: PairlistData,
    ) -> Dict[str, torch.Tensor]:
        """
        Implements the forward pass through the network.

        Parameters
        ----------
        data : NNPInput
            Contains input data for the batch obtained directly from the
            dataset, including atomic numbers, positions, and other relevant
            fields.
        pairlist_output : PairListOutputs
            Contains the indices for the selected pairs and their associated
            distances and displacement vectors.

        Returns
        -------
        Dict[str, torch.Tensor]
            The calculated per-atom properties and other properties from the
            forward pass.
        """
        # perform the forward pass implemented in the subclass
        results = self.compute_properties(data, pairlist_output)
        atomic_embedding = results["per_atom_scalar_representation"]

        # The way this is setup, we will always return per_atom_charge and electrostatic_energy
        # as those come directly from the compute_properties method.

        # Compute all specified outputs
        for output_name, output_layer in self.output_layers.items():
            output = output_layer(atomic_embedding)
            results[output_name] = output

        return results

In [14]:
multipole_aimnet = MultipoleRep_AimNet2Core(
    featurization={
        "properties_to_featurize": ["atomic_number"],
        "atomic_number": {
            "maximum_atomic_number": 101,
            "number_of_per_atom_features": 6,
        },
    },
    number_of_radial_basis_functions=64,
    number_of_vector_features=128,
    number_of_interaction_modules=3,
    interaction_module_hidden_layers=[[512,380], [512,380], [512,380, 380]],
    output_module_hidden_layers=[128, 128],
    activation_function_parameter={
        "activation_function_name": "SiLU",
        "activation_function": nn.SiLU(),
    },
    predicted_properties=["per_atom_energy"],
    predicted_dim=[1],
    maximum_interaction_radius=0.5,
)

2026-10-25 16:22:25.435 | DEBUG    | __main__:__init__:52 - Initializing the AimNet2 architecture.


In [15]:
multipole_aimnet(data, neighborlist.local_cutoff)

{'per_atom_scalar_representation': tensor([[ 0.2317,  0.0224,  4.9626,  1.6052, -0.0289, -0.0107],
         [ 0.4172,  1.4544,  4.3580,  1.2645,  0.0308,  0.8145],
         [ 0.4172,  1.4544,  4.3580,  1.2645,  0.0308,  0.8145]],
        grad_fn=<AddBackward0>),
 'atomic_subsystem_indices': tensor([0, 0, 0]),
 'atomic_numbers': tensor([26,  8,  8]),
 'per_atom_charge': tensor([[0.3333],
         [0.3333],
         [0.3333]], grad_fn=<AddBackward0>),
 'per_atom_energy': tensor([[-0.1081],
         [-0.0717],
         [-0.0717]], grad_fn=<AddmmBackward0>)}

## Test cases

### Test representations

In [16]:
representation_module = core.representation_module

# two systems
# 3 atoms, 4 atoms
multipole_input = torch.tensor(
    [
        [3, 1, 2, 3, 1, 2, 3, 4, 5],
        [4, 1, 2, 3, 1, 2, 3, 4, 5],
        [5, 1, 2, 3, 1, 2, 3, 4, 5],
        [6, 1, 2, 3, 1, 2, 3, 4, 5],
        [6, 1, 2, 3, 1, 2, 3, 4, 5],
        [9, 1, 2, 3, 1, 2, 3, 4, 5],
        [14, 1, 2, 3, 1, 2, 3, 4, 5],
    ],
    dtype=torch.float32,
)
monopole_target = torch.tensor([-1, 2])
atomic_subsystem_indices = torch.tensor([0, 0, 0, 1, 1, 1, 1])

scaled_multipole = representation_module._scale_monopole(
    multipole_input,
    monopole_target,
    atomic_subsystem_indices
)

scaled_multipole_true = torch.tensor(
    [
        [-0.25, 1, 2, 3, 1, 2, 3, 4, 5],
        [-0.3333, 1, 2, 3, 1, 2, 3, 4, 5],
        [-0.4167, 1, 2, 3, 1, 2, 3, 4, 5],
        [0.3429, 1, 2, 3, 1, 2, 3, 4, 5],
        [0.3429, 1, 2, 3, 1, 2, 3, 4, 5],
        [0.5143, 1, 2, 3, 1, 2, 3, 4, 5],
        [0.8, 1, 2, 3, 1, 2, 3, 4, 5],
    ],
    dtype=torch.float32,
)

assert torch.allclose(scaled_multipole, scaled_multipole_true, atol=1e-4)

### A batch input containing 3 systems

In [17]:
def setup_test_system_nnp_input():

    # 12 atoms in 3 systems
    atomic_numbers = torch.tensor([
        26, 8, 8, 29, 17, 17, 30, 16, 8, 8, 8, 8
    ])
    atomic_subsystem_indices = torch.tensor([
        0, 0, 0, 1, 1, 1, 2, 2, 2, 2, 2, 2
    ])

    # random positions for 12 atoms
    positions = torch.rand(12, 3)

    per_system_total_charge = torch.tensor([-1, 0, 1])
    per_system_spin_multiplicity = torch.tensor([1, 3, 5])

    data = NNPInput(
        atomic_numbers=atomic_numbers,
        positions=positions,
        atomic_subsystem_indices=atomic_subsystem_indices,
        per_system_total_charge=per_system_total_charge,
        per_system_spin_multiplicity=per_system_spin_multiplicity,
    )

    neighborlist_module = NeighborListForTraining(
        local_cutoff=5,
    )
    neighborlist = neighborlist_module(
        data,
    )

    return data, neighborlist

### Rotation invariance and equivariance

In [95]:
nnp_input, neighborlist = setup_test_system_nnp_input()

In [96]:
# Create a rotation matrix for the test system with size (number_of_atoms, 3, 3)
number_of_systems = int(nnp_input.atomic_subsystem_indices.max()) + 1
rotation = o3.rand_matrix(number_of_systems)
d_matrix = o3.Irreps("1x0e + 1x1o + 1x2e").D_from_matrix(rotation)
per_atom_rotation = rotation[nnp_input.atomic_subsystem_indices]
per_atom_d_matrix = d_matrix[nnp_input.atomic_subsystem_indices]

In [97]:
rotation.shape

torch.Size([3, 3, 3])

In [98]:
per_atom_rotation.shape

torch.Size([12, 3, 3])

In [99]:
d_matrix.shape

torch.Size([3, 9, 9])

In [100]:
per_atom_d_matrix.shape

torch.Size([12, 9, 9])

In [104]:
# rotate input coordinates and neighbor list
rotated_nnp_input = copy.deepcopy(nnp_input)
rotated_nnp_input.positions = torch.einsum(
    "nij,nj->ni",
    per_atom_rotation,
    nnp_input.positions,
)

neighborlist_module = NeighborListForTraining(
    local_cutoff=5,
)
rotated_neighborlist = neighborlist_module(
    rotated_nnp_input,
)

In [105]:
# charge_dipole from the original input
output = core(nnp_input, neighborlist.local_cutoff)

# rotate the charge multipole
charge_multipole = output["per_atom_charge_multipole"]
rotated_charge_multipole = torch.einsum(
    "nij,nj->ni",
    per_atom_d_matrix,
    charge_multipole,
)

In [106]:
# charge dipole from the rotated input
rotated_output = core(rotated_nnp_input, rotated_neighborlist.local_cutoff)

In [118]:
torch.allclose(
    rotated_charge_multipole,
    rotated_output["per_atom_charge_multipole"],
    atol=1e-4,
)

True

In [119]:
torch.allclose(
    rotated_output["per_atom_energy"],
    output["per_atom_energy"],
    atol=1e-4,
)

True